In [ ]:
import triton
import SupertracePybind as Supertrace
from supertrace_util import compatibleProcessing, initTritonCtxEnv, mergeRepeatIns, checkIndirectIns

In [ ]:
tracepath = "success.trace64"
tracepath = "failed.trace64"

trace = Supertrace.parse_x64dbg_trace(tracepath)
record = trace.getRecord() # 获取trace的记录指令列表
print(f"trace instruction num: {len(record)}")

In [ ]:
modules = trace.user.meta.getModules()
for mod in modules:
    if mod.isMainModule:
        main_module = mod
        break

main_secs = main_module.getSections()
print("No\tName")
for i, sec in enumerate(main_secs):
    print(f"{i}\t{sec.name}")

In [ ]:
vmpsec_begin = main_secs[1].addr
vmpsec_end = main_secs[5].addr
print(f"vmp begin: {hex(vmpsec_begin)}")
print(f"vmp end: {hex(vmpsec_end)}")

In [ ]:
ctx = triton.TritonContext()
ctx.setArchitecture(triton.ARCH.X86_64)
ctx.setMode(triton.MODE.ALIGNED_MEMORY, True)
ctx.setMode(triton.MODE.AST_OPTIMIZATIONS, True)
ctx.setMode(triton.MODE.CONSTANT_FOLDING, True)
ctx.setMode(triton.MODE.ONLY_ON_TAINTED, True)
ctx.setMode(triton.MODE.TAINT_THROUGH_POINTERS, True) # 开启内存指针污染

In [ ]:
record = mergeRepeatIns(ctx, record, True) # 合并trace里的rep指令
print(f"after mergeing 'rep' instructions, trace instruction num: {len(record)}")

In [ ]:
memMaps = trace.user.meta.getMemoryMaps()
anyStackAddr = record[0].reg_dump64.regcontext.csp
stackMemArea = None
for mm in memMaps:
    if (mm.addr + mm.size >= anyStackAddr >= mm.addr):
        stackMemArea = mm
        break
print(f"stack begin: {hex(stackMemArea.addr)}")
print(f"stack end: {hex(stackMemArea.addr + stackMemArea.size)}")

In [ ]:
threads = trace.user.meta.getThreads()
for th in threads:
    if th.id == record[0].thread_id:
        main_thread = th
        break
print(f"main thread id: {main_thread.id} ({hex(main_thread.id)})")
print(f"teb: {hex(main_thread.teb)}") # 获取线程TEB地址
initTritonCtxEnv(ctx, record[0], main_thread.teb) # 初始化寄存器环境

In [ ]:
handlers: list[list[Supertrace.InstructionRecord]] = []
handler: list[Supertrace.InstructionRecord] = []

In [ ]:
for i, ins in enumerate(record):
    if (i + 1 >= len(record)): nextIns = None
    else: nextIns = record[i + 1]

    ttins = triton.Instruction()
    ttins.setAddress(ins.ins_address)
    ttins.setOpcode(ins.bytes)
    ctx.disassembly(ttins)

    ins.ttins = ttins

    for memAcc in ins.mem_accs:
        if (memAcc.type == Supertrace.AccessType.READ and (vmpsec_begin <= memAcc.acc_address <= vmpsec_end)):
            ctx.taintMemory(triton.MemoryAccess(memAcc.acc_address, memAcc.acc_size)) # 污染来自VM区块的字节码（确保不会把混淆间接跳转也给识别进去）

    handler.append(ins)

    compatibleProcessing(ctx, ttins, ins, nextIns, True, False) # 执行指令

    if (checkIndirectIns(ttins) and (nextIns is not None) and ttins.isTainted()):
        handlers.append(handler.copy())
        handler.clear()

In [ ]:
# 去掉vmenter和vmexit
del handlers[0]
# del handlers[-1]

In [ ]:
all_handlers = set()

In [ ]:
for i, handler in enumerate(handlers):
    firstIns = handler[0]
    all_handlers.add(firstIns.ins_address)

In [ ]:
print(f"去重前的总handler数量为: {len(handlers)}")

In [ ]:
print(f"去重后的总handler数量为: {len(all_handlers)}")

In [ ]:
def getMask(n):
    if (n == 1):
        return 0xff
    elif (n == 2):
        return 0xffff
    elif (n == 4):
        return 0xffffffff
    elif (n == 8):
        return 0xffffffffffffffff
    else:
        raise ValueError

In [ ]:
for i, handler in enumerate(handlers):
    firstIns = handler[0]

    tmpctx = triton.TritonContext()
    tmpctx.setArchitecture(triton.ARCH.X86_64)
    tmpctx.setMode(triton.MODE.ALIGNED_MEMORY, True)
    tmpctx.setMode(triton.MODE.AST_OPTIMIZATIONS, True)
    tmpctx.setMode(triton.MODE.CONSTANT_FOLDING, True)
    tmpctx.setMode(triton.MODE.TAINT_THROUGH_POINTERS, True)

    initTritonCtxEnv(tmpctx, firstIns, main_thread.teb)

    if (firstIns.dbg_id >= 0x35e9):
        continue

    vspRegName = "r8"
    vpcRegName = "rdi"
    vkeyRegName = "r9"
    vbaseRegName = "rbx"

    # vspRegName = "rsi"
    # vpcRegName = "rbx"
    # vkeyRegName = "rdi"
    # vbaseRegName = "rbp"
    vspReg = getattr(tmpctx.registers, vspRegName)
    vpcReg = getattr(tmpctx.registers, vpcRegName)
    vkeyReg = getattr(tmpctx.registers, vkeyRegName)
    vbaseReg = getattr(tmpctx.registers, vbaseRegName)
    tmpctx.taintRegister(vspReg)
    tmpctx.taintRegister(vpcReg)
    tmpctx.taintRegister(vkeyReg)
    tmpctx.taintRegister(vbaseReg)

    initVsp = tmpctx.getConcreteRegisterValue(vspReg)
    initRsp = firstIns.reg_dump64.regcontext.csp

    print(f"--[{i}]---------- {hex(firstIns.dbg_id)} addr: {hex(firstIns.ins_address)} --------------")
    print(f"|\tvsp[{vspRegName}]: {hex(tmpctx.getConcreteRegisterValue(vspReg))[2:]}    vpc[{vpcRegName}]: {hex(tmpctx.getConcreteRegisterValue(vpcReg))[2:]}    rsp: {hex(firstIns.reg_dump64.regcontext.csp)[2:]}    vkey[{vkeyRegName}]: {hex(tmpctx.getConcreteRegisterValue(vkeyReg))[2:]}    vbase[{vbaseRegName}]: {hex(tmpctx.getConcreteRegisterValue(vbaseReg))[2:]}")

    memInfo = []

    for ii, ins in enumerate(handler):
        if (ii + 1 >= len(handler)): nextIns = None
        else: nextIns = handler[ii + 1]

        ttins = triton.Instruction()
        ttins.setAddress(ins.ins_address)
        ttins.setOpcode(ins.bytes)

        for memAcc in ins.mem_accs:
            if (memAcc.type != Supertrace.AccessType.READ):
                continue
            for i in range(memAcc.acc_size):
                memi = triton.MemoryAccess(memAcc.acc_address + i, triton.CPUSIZE.BYTE)
                if (not tmpctx.isConcreteMemoryValueDefined(memi)):
                    oldby = (memAcc.old_data >> (i * 8)) & 0xFF
                    tmpctx.setConcreteMemoryValue(memi, oldby)

        tmpctx.processing(ttins)

        insType = ttins.getType()
        if (insType == triton.OPCODE.X86.CPUID):
            tmpctx.taintRegister(tmpctx.registers.rax)
            tmpctx.taintRegister(tmpctx.registers.rbx)
            tmpctx.taintRegister(tmpctx.registers.rcx)
            tmpctx.taintRegister(tmpctx.registers.rdx)
        elif (insType == triton.OPCODE.X86.RDTSC):
            tmpctx.taintRegister(tmpctx.registers.rax)
            tmpctx.taintRegister(tmpctx.registers.rdx)
        elif (insType == triton.OPCODE.X86.RDTSCP):
            tmpctx.taintRegister(tmpctx.registers.rax)
            tmpctx.taintRegister(tmpctx.registers.rcx)
            tmpctx.taintRegister(tmpctx.registers.rdx)
        elif (insType == triton.OPCODE.X86.RDSEED or insType == triton.OPCODE.X86.RDRAND):
            opreg: triton.Register = ttins.getOperands()[0]
            tmpctx.taintRegister(opreg)

        show = False
        if (insType == triton.OPCODE.X86.LEA and not ttins.isTainted()): # 与rsp有关的地址lea指令寻址
            readRegs = ttins.getReadRegisters()
            for ttreg, ast in readRegs:
                if (tmpctx.getParentRegister(ttreg).getId() == triton.REG.X86_64.RSP):
                    regop: triton.Register = ttins.getOperands()[0]
                    show = True
                    if (tmpctx.getParentRegister(regop).getId() != triton.REG.X86_64.RSP): # 比如 lea rdx, qword ptr ss:[rsp+0x20]
                        tmpctx.taintRegister(regop)
                    break

        if (ttins.isTainted() or show or insType == triton.OPCODE.X86.CPUID or insType == triton.OPCODE.X86.RDTSC or insType == triton.OPCODE.X86.RDTSCP or insType == triton.OPCODE.X86.RDSEED or insType == triton.OPCODE.X86.RDRAND):
            asm = str(ttins)
            asm = asm.replace(" + riz", "") # 针对内存操作数，删除'riz'的字眼

            for memacc in ins.mem_accs: # 读取了vm字节码就在前面空一行
                if (vmpsec_begin <= memacc.acc_address <= vmpsec_end):
                    print("|")
            
            changed_vspOrvpc = " "
            for ttreg, astNode in ttins.getWrittenRegisters(): # 修改了vsp、vpc、vbase或vkey就在前面加个*号
                if (tmpctx.getParentRegister(ttreg).getName() in [vspRegName, vpcRegName, vbaseRegName, vkeyRegName]):
                    changed_vspOrvpc = "*"
            
            # 跳转下一个handler的代码部分也空行
            if (ii == len(handler) - 1 and ttins.getType() == triton.OPCODE.X86.JMP): # "jmp reg"形式
                print(f"|")
            elif (ii == len(handler) - 2 and ttins.getType() == triton.OPCODE.X86.MOV and handler[ii + 1].ttins.getType() == triton.OPCODE.X86.RET): # "mov [rsp], reg; ret"形式
                print(f"|")

            print(f"|{changed_vspOrvpc}{hex(ins.dbg_id)} {asm}")

            for ttreg, astNode in ttins.getWrittenRegisters(): # 修改了vkey就空一行
                if (tmpctx.getParentRegister(ttreg).getName() in [vkeyRegName]):
                    print("|")

            for memacc in ins.mem_accs:
                mask = getMask(memacc.acc_size)

                vrInfo = ""
                VRLIMIT = 0x100 # 该值需从checkVsp的修正的ecx值里获取
                if ((initRsp + VRLIMIT > memacc.acc_address >= initRsp) and (memacc.acc_address & 7) == 0): # 可能的虚拟寄存器寻址
                    off = (memacc.acc_address - initRsp)
                    vrN = off // 8
                    vrInfo = f" [+{hex(off)} = VR{vrN}]"

                vspInfo = ""
                if ((stackMemArea.addr + stackMemArea.size) >= memacc.acc_address >= (initRsp + VRLIMIT)): # 可能的虚拟栈寻址
                    off = memacc.acc_address - initVsp # 偏移都基于handler首部时刻的vsp开始计算
                    if (off == 0):
                        vspInfo = f" vsp[+0x0]"
                    elif (off > 0):
                        vspInfo = f" vsp[+{hex(off)}]"
                    else: # off < 0
                        vspInfo = f" vsp[{hex(off)}]"

                if (memacc.type == Supertrace.AccessType.READ):
                    memInfo.append(f"| {hex(ins.dbg_id)} [R]{vrInfo}{vspInfo} {hex(memacc.acc_address)[2:]}:\t{hex(memacc.new_data & mask)[2:]}")
                elif (memacc.type == Supertrace.AccessType.WRITE):
                    memInfo.append(f"| {hex(ins.dbg_id)} [W]{vrInfo}{vspInfo} {hex(memacc.acc_address)[2:]}:\t{hex(memacc.old_data & mask)[2:]} -> {hex(memacc.new_data & mask)[2:]}")

    if (len(memInfo) > 0):
        print("--------------------------")
        for memI in memInfo:
            print(memI)

    print()
    print()